In [ ]:
import json
import logging
import pandas as pd
import numpy as np
import uuid
import getpass
import os
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Set

# Config

In [ ]:
# Required: Atlas table with parameters
table_file_path = "/global/homes/b/bkieft/metatlas-data/HILIC/HILIC_QCv7_positive.tsv"
databases_path = "/global/homes/b/bkieft/metatlas2/data/databases"

# Required: Method information for this atlas
atlas_type = "qc" # "istd", "qc", or "ema"
atlas_name = "HILIC_QCv7_positive"
atlas_description = ""
chromatography = "HILIC"  # e.g., "HILIC", "C18", "PFPP"
polarity = "positive"     # "positive" or "negative"
method_name = f"{chromatography}_{polarity}"

# Optional: Additional method details
column_details = "HILIC column"
mobile_phase = "ACN/H2O gradient"
instrument = "QExactive"
method_notes = "Standard HILIC method for metabolomics"

# Generate unique atlas UID for this atlas
atlas_uid = f"{atlas_type}-atlas-{chromatography}-{polarity}-{uuid.uuid4().hex[:16]}"

# Required: Path to core compounds database
compounds_database_dir = f"{databases_path}/compounds"

# Required: Output directory for reference databases
references_database_dir = f"{databases_path}/mz_rt_references"

# Required: Output directory for atlas files
output_dir = f"{databases_path}/atlases"

# Optional: RT and MZ tolerances for atlas entries (override table values if needed)
override_rt_tolerance = False  # Use RT bounds from table file
override_mz_tolerance = False  # Use MZ tolerance from table file
default_rt_tolerance = 0.5  # minutes (used if table doesn't have rt_min/rt_max)
default_mz_tolerance = 5.0  # ppm (used if table doesn't have mz_tolerance)

# Optional: Validation settings
require_rt_data = True   # Skip compounds without RT data in table
require_mz_data = True   # Skip compounds without MZ data in table
require_compound_in_db = True  # Skip compounds not found in database

# Optional: Force creation of new entries even if they exist
force_new_entry = False  # Set to True to always create new MZ RT references

# Creator information
creator_name = getpass.getuser()
creation_notes = ""
timestamp = datetime.now().isoformat()
create_backup = True

# Checks
if atlas_type not in ["istd", "qc", "ema"]:
    raise ValueError(f"Invalid atlas type: {atlas_type}. Must be one of ['istd', 'qc', 'ema']")

print(f"Atlas UID: {atlas_uid}")
print(f"Atlas type: {atlas_type}")
print(f"Input table: {table_file_path}")
print(f"Compounds DB: {compounds_database_dir}")
print(f"References DB: {references_database_dir}")
print(f"Output: {output_dir}")
print(f"Method: {chromatography} {polarity}")
print(f"Current user: {creator_name}")
print(f"Timestamp: {timestamp}")

# Helper functions

In [ ]:
# Database helper functions
def build_mz_rt_index(mz_rt_references):
    """Build index mapping compound_uid to list of mz_rt_reference_uids"""
    index = {"compound_to_mz_rt_refs": {}}
    
    for mz_rt_ref_uid, mz_rt_data in mz_rt_references.items():
        compound_uid = mz_rt_data.get('compound_uid')
        if compound_uid:
            if compound_uid not in index["compound_to_mz_rt_refs"]:
                index["compound_to_mz_rt_refs"][compound_uid] = []
            index["compound_to_mz_rt_refs"][compound_uid].append(mz_rt_ref_uid)
    
    return index

def find_existing_mz_rt_reference(compound_uid, method_uid, mz_rt_references, mz_rt_index):
    """Find existing MZ RT reference for compound and method combination"""
    compound_refs = mz_rt_index.get("compound_to_mz_rt_refs", {}).get(compound_uid, [])
    
    for ref_uid in compound_refs:
        ref_data = mz_rt_references.get(ref_uid, {})
        if ref_data.get('method_uid') == method_uid:
            return ref_uid, ref_data
    
    return None, None

def find_or_create_method(methods_db, method_config):
    """Find existing method or create new one"""
    # Look for existing method with same chromatography and polarity
    for method_uid, method_data in methods_db.items():
        if (method_data.get('chromatography') == method_config['chromatography'] and
            method_data.get('polarity') == method_config['polarity']):
            print(f"Found existing method: {method_uid} ({method_data.get('method_name')})")
            return method_uid
    
    # Create new method
    method_uid = f"method-{uuid.uuid4().hex[:16]}"
    methods_db[method_uid] = {
        "method_uid": method_uid,
        "method_name": method_config['method_name'],
        "chromatography": method_config['chromatography'],
        "polarity": method_config['polarity'],
        "column": method_config.get('column', ''),
        "mobile_phase": method_config.get('mobile_phase', ''),
        "instrument": method_config.get('instrument', ''),
        "method_notes": method_config.get('method_notes', ''),
        "created_by": creator_name,
        "creation_time": timestamp,
        "last_modified": timestamp
    }
    
    print(f"Created new method: {method_uid} ({method_config['method_name']})")
    return method_uid

def create_mz_rt_reference(compound_uid, compound_name, method_uid, rt_data, mz_data):
    """Create combined MZ_RT reference entry"""
    mz_rt_uid = f"mzrt-ref-{uuid.uuid4().hex[:16]}"
    
    mz_rt_entry = {
        "mz_rt_reference_uid": mz_rt_uid,
        "compound_uid": compound_uid,
        "compound_name": compound_name,
        "method_uid": method_uid,
        
        # RT data
        "rt_peak": rt_data['rt_peak'],
        "rt_min": rt_data['rt_min'],
        "rt_max": rt_data['rt_max'],
        "rt_units": rt_data.get('rt_units', 'min'),
        
        # MZ data
        "mz": mz_data['mz'],
        "mz_tolerance": mz_data['mz_tolerance'],
        "mz_tolerance_units": mz_data.get('mz_tolerance_units', 'ppm'),
        "adduct": mz_data['adduct'],
        "charge": mz_data.get('charge', 0),
        
        # Metadata
        "confidence": rt_data.get('confidence', 'experimental'),
        "source": rt_data.get('source', 'table_input'),
        "created_by": creator_name,
        "creation_time": timestamp,
        "last_modified": timestamp
    }
    
    return mz_rt_uid, mz_rt_entry

def create_database_backups(reference_files, timestamp):
    """Create backups of existing reference database files in a 'backups' subfolder with timestamped filenames."""
    backup_files = {}
    for db_name, file_path in reference_files.items():
        file_path_obj = Path(file_path)
        if file_path_obj.exists():
            # Create backups directory inside the file's parent directory
            backup_dir = file_path_obj.parent / "backups"
            backup_dir.mkdir(parents=True, exist_ok=True)
            # Create backup filename with timestamp
            backup_filename = f"{file_path_obj.stem}_backup_{timestamp}{file_path_obj.suffix}"
            backup_path = backup_dir / backup_filename
            # Copy the file to backup location
            file_path_obj.replace(backup_path)
            backup_files[db_name] = str(backup_path)
            print(f"Created backup for {db_name}: {backup_path}")
    return backup_files

def save_reference_databases(reference_databases, reference_files):
    """Save all reference databases to files"""
    # Rebuild index before saving
    reference_databases['mz_rt_index'] = build_mz_rt_index(reference_databases['mz_rt_references'])
    
    for db_name, data in reference_databases.items():
        file_path = reference_files[db_name]
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"Saved {db_name}: {len(data)} entries to {file_path}")

# Load Databases and Create Reference Database Functions

In [ ]:
# Load core compounds database
compounds_file = Path(compounds_database_dir) / 'Compounds.json'
compound_index_file = Path(compounds_database_dir) / 'Compounds_Index.json'

# Check required files exist
if not compounds_file.exists():
    raise FileNotFoundError(f"Compounds database not found: {compounds_file}")
if not compound_index_file.exists():
    raise FileNotFoundError(f"Compound index not found: {compound_index_file}")

print("Loading compounds database...")

# Load compounds database
with open(compounds_file, 'r') as f:
    compounds_db = json.load(f)

# Load compound index
with open(compound_index_file, 'r') as f:
    compound_index = json.load(f)

print("Database loaded successfully!")
print(f"  Compounds: {len(compounds_db)}")
print(f"  InChI keys indexed: {len(compound_index.get('compound_by_inchi_key', {}))}")

# Create references database directory
references_path = Path(references_database_dir)
references_path.mkdir(parents=True, exist_ok=True)

# Define reference database files - now using combined MZ_RT_References
reference_files = {
    'methods': references_path / 'MZ_RT_References_Methods.json',
    'mz_rt_references': references_path / 'MZ_RT_References.json',
    'mz_rt_index': references_path / 'MZ_RT_References_Index.json'
}

# Load existing reference databases or create empty ones
reference_databases = {}
for db_name, file_path in reference_files.items():
    if file_path.exists():
        with open(file_path, 'r') as f:
            reference_databases[db_name] = json.load(f)
        print(f"Loaded {db_name}: {len(reference_databases[db_name])} entries")
    else:
        reference_databases[db_name] = {}
        print(f"Created new {db_name} database")
    reference_files[db_name] = str(file_path)

# Load and validate table data
print(f"Loading table data from: {table_file_path}")
delimiter = '\t' if table_file_path.endswith(('.tsv', '.tab', '.txt')) else ','
table_data = pd.read_csv(table_file_path, sep=delimiter)
print(f"Loaded {len(table_data)} rows from table")

# Check required columns
required_columns = ['inchi_key', 'label', 'rt_peak', 'mz', 'adduct']
missing_columns = [col for col in required_columns if col not in table_data.columns]
if missing_columns:
    raise ValueError(f"table missing required columns: {missing_columns}")

# Process table Data and Create Reference Databases

In [ ]:
# Find or create method for this dataset
method_config = {
    'method_name': method_name,
    'chromatography': chromatography,
    'polarity': polarity,
    'column': column_details,
    'mobile_phase': mobile_phase,
    'instrument': instrument,
    'method_notes': method_notes
}

current_method_uid = find_or_create_method(reference_databases['methods'], method_config)

# Build or rebuild the MZ RT index
print("Building MZ RT reference index...")
reference_databases['mz_rt_index'] = build_mz_rt_index(reference_databases['mz_rt_references'])

# Process table data and create reference databases
print("Processing table data and creating reference databases...")

atlas_compounds = []
processing_stats = {
    "input_compounds": len(table_data),
    "compounds_found_in_db": 0,
    "compounds_missing_from_db": 0,
    "compounds_missing_rt": 0,
    "compounds_missing_mz": 0,
    "mz_rt_references_created": 0,
    "mz_rt_references_existing": 0,
    "compounds_added_to_atlas": 0
}

missing_compounds = []
for idx, row in table_data.iterrows():
    inchi_key = row['inchi_key']
    compound_name = row.get('label', '')
    
    if pd.isna(inchi_key) or inchi_key == '':
        print(f"Row {idx}: Missing InChI key, skipping")
        continue
    
    # Look up compound in database
    compound_uids = compound_index.get('compound_by_inchi_key', {}).get(inchi_key, [])
    
    if not compound_uids:
        processing_stats["compounds_missing_from_db"] += 1
        missing_compounds.append(inchi_key)
        if require_compound_in_db:
            print(f"Row {idx}: InChI key {inchi_key[:20]}... not found in database, skipping")
            continue
    else:
        processing_stats["compounds_found_in_db"] += 1
        # Use the most recent compound if multiple exist
        compound_uid = compound_uids[-1] if isinstance(compound_uids, list) else compound_uids
    
    # Validate required data
    if require_rt_data and (pd.isna(row.get('rt_peak')) or row.get('rt_peak') == ''):
        processing_stats["compounds_missing_rt"] += 1
        print(f"Row {idx}: Missing RT data for {inchi_key[:20]}..., skipping")
        continue
        
    if require_mz_data and (pd.isna(row.get('mz')) or row.get('mz') == ''):
        processing_stats["compounds_missing_mz"] += 1
        print(f"Row {idx}: Missing MZ data for {inchi_key[:20]}..., skipping")
        continue
    
    # Check for existing MZ RT reference unless forcing new entries
    mz_rt_ref_uid = None
    if not force_new_entry and compound_uids:
        existing_ref_uid, existing_ref_data = find_existing_mz_rt_reference(
            compound_uid, current_method_uid, 
            reference_databases['mz_rt_references'], 
            reference_databases['mz_rt_index']
        )
        
        if existing_ref_uid:
            mz_rt_ref_uid = existing_ref_uid
            processing_stats["mz_rt_references_existing"] += 1
    
    # Create new MZ RT reference if none exists or forcing new entries
    if mz_rt_ref_uid is None:
        # Prepare RT data
        rt_data = {
            "rt_peak": float(row['rt_peak']),
            "rt_min": float(row.get('rt_min', row['rt_peak'] - default_rt_tolerance)),
            "rt_max": float(row.get('rt_max', row['rt_peak'] + default_rt_tolerance)),
            "rt_units": str(row.get('rt_units', 'min')),
            "confidence": "atlas",
            "source": table_file_path
        }
        
        # Override RT tolerance if specified
        if override_rt_tolerance:
            rt_data["rt_min"] = rt_data["rt_peak"] - default_rt_tolerance
            rt_data["rt_max"] = rt_data["rt_peak"] + default_rt_tolerance
        
        # Prepare MZ data
        adduct = str(row.get('adduct', 'unknown'))
        charge = 1 if '+' in adduct else -1 if '-' in adduct else 0
        
        mz_data = {
            "mz": float(row['mz']),
            "mz_tolerance": float(row.get('mz_tolerance', default_mz_tolerance)),
            "mz_tolerance_units": str(row.get('mz_tolerance_units', 'ppm')),
            "adduct": adduct,
            "charge": charge,
            "confidence": "atlas",
            "source": table_file_path
        }
        
        # Override MZ tolerance if specified
        if override_mz_tolerance:
            mz_data["mz_tolerance"] = default_mz_tolerance
        
        # Create combined MZ_RT reference
        mz_rt_ref_uid, mz_rt_ref_entry = create_mz_rt_reference(compound_uid, compound_name, current_method_uid, rt_data, mz_data)
        reference_databases['mz_rt_references'][mz_rt_ref_uid] = mz_rt_ref_entry
        processing_stats["mz_rt_references_created"] += 1
        
        # Update index
        if compound_uid not in reference_databases['mz_rt_index'].get("compound_to_mz_rt_refs", {}):
            reference_databases['mz_rt_index'].setdefault("compound_to_mz_rt_refs", {})[compound_uid] = []
        reference_databases['mz_rt_index']["compound_to_mz_rt_refs"][compound_uid].append(mz_rt_ref_uid)
    
    # Create atlas compound entry
    atlas_compound = {
        "compound_uid": compound_uid if compound_uids else f"missing_{inchi_key[:16]}",
        "mz_rt_reference_uid": mz_rt_ref_uid
    }
    
    atlas_compounds.append(atlas_compound)
    processing_stats["compounds_added_to_atlas"] += 1

print(f"Processing complete!")
print(f"  Table rows processed: {processing_stats['input_compounds']}")
print(f"  Compounds found in database: {processing_stats['compounds_found_in_db']}")
print(f"  Compounds missing from database: {processing_stats['compounds_missing_from_db']}")
print(f"  MZ_RT references created: {processing_stats['mz_rt_references_created']}")
print(f"  MZ_RT references reused: {processing_stats['mz_rt_references_existing']}")
print(f"  Compounds added to atlas: {processing_stats['compounds_added_to_atlas']}")

if missing_compounds:
    print(f"Missing compounds (first 5): {missing_compounds[:5]}")
    if len(missing_compounds) > 5:
        print(f"  ... and {len(missing_compounds) - 5} more")

if not atlas_compounds:
    raise ValueError("No valid atlas compounds created. Check your table data and database.")

if processing_stats['mz_rt_references_existing'] > 0:
    print(f'Note! Used existing MZ RT references for {processing_stats["mz_rt_references_existing"]} compounds.')

# Create the atlas object
print("Creating atlas object...")

atlas = {
    "atlas_uid": atlas_uid,
    "atlas_name": atlas_name,
    "atlas_description": atlas_description,
    "created_by": creator_name,
    "creation_time": timestamp,
    "last_modified": timestamp,
    "notes": creation_notes,
    
    # Method information embedded in atlas
    "method": {
        "method_name": method_name,
        "chromatography": chromatography,
        "polarity": polarity
    },
    
    # Atlas-specific settings
    "atlas_settings": {
        "source_table": str(table_file_path),
        "compounds_database": str(compounds_file),
        "total_compounds": len(atlas_compounds),
        "default_mz_tolerance_ppm": default_mz_tolerance
    },

    # Source of RT and MZ data
    "rt_mz_source": reference_files,

    # Processing statistics
    "processing_stats": processing_stats,
    
    # Embedded compound data with experimental references
    "compounds": atlas_compounds
}

print(f"Atlas created successfully!")
print(f"  Atlas UID: {atlas_uid}")
print(f"  Atlas name: {atlas_name}")
print(f"  Method: {chromatography} {polarity}")
print(f"  Total compounds: {len(atlas_compounds)}")

# Always create new atlas (no backup needed)
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

atlas_filename = f"{atlas_uid}.json"
atlas_file_path = output_path / atlas_filename

print(f"Saving new atlas to: {atlas_file_path}")
with open(atlas_file_path, 'w') as f:
    json.dump(atlas, f, indent=2)

# Calculate file size
file_size_kb = atlas_file_path.stat().st_size / 1024
print(f"Atlas saved successfully! ({file_size_kb:.1f} KB)")

if processing_stats['mz_rt_references_created'] > 0:
    if create_backup:
        existing_files = {name: path for name, path in reference_files.items() 
                        if Path(path).exists()}
        if existing_files:
            print("Creating backups of existing reference database files...")
            backup_files = create_database_backups(existing_files, timestamp)
            print(f"Created {len(backup_files)} backup files")
        else:
            print("No existing reference files to backup")
    print("Saving reference databases...")
    save_reference_databases(reference_databases, reference_files)
else:
    print("No new MZ RT references created, skipping database save and backup.")

In [ ]:
# Validation and summary
print("\n Atlas Summary:")
print(f"  Atlas file: {atlas_filename}")
print(f"  Method UID: {current_method_uid}")
print(f"  Total compounds: {len(atlas_compounds)}")

# Get method details
method_details = reference_databases['methods'][current_method_uid]
print(f"  Method: {method_details['chromatography']} {method_details['polarity']}")

# Database statistics
print(f"\n Reference Database Summary:")
print(f"  Methods: {len(reference_databases['methods'])}")
print(f"  MZ_RT References: {len(reference_databases['mz_rt_references'])}")
print(f"  Compound-to-MZ_RT mappings: {len(reference_databases['mz_rt_index'].get('compound_to_mz_rt_refs', {}))}")
print(f"  New compounds added to MZ_RT References: {processing_stats['mz_rt_references_created']}")

print(f"\n Atlas and reference databases created successfully!")
print(f" Atlas file: {atlas_file_path}")
print(f" Reference databases: {references_database_dir}")